## Download Divvy-Tripdata

This notebook implements a reproducible data ingestion pipeline for Divvy bike-share trip data, aligned with the **CRISP-DM** process. It automates the retrieval of monthly trip files from the public Divvy S3 bucket, filters them by a configurable date range, downloads the corresponding ZIP archives, extracts their contents, and consolidates all CSVs into a single master directory for downstream analysis and modeling.

Within CRISP-DM, this notebook supports the *Data Understanding* and *Data Preparation* phases by ensuring that raw source data is consistently collected, versioned by month, and organized into a clean file structure. The resulting CSV repository can be used for exploratory data analysis, feature engineering, and machine learning experiments in subsequent notebooks.

### Import libraries

In [1]:
# Standard library modules
import os                      # Work with file paths and directories
import requests                # Send HTTP requests to download data from the web
from bs4 import BeautifulSoup  # Parse XML/HTML responses from the Divvy S3 bucket
from datetime import datetime  # Get the current date to limit downloads to available months
from tqdm import tqdm          # Display progress bars for long-running loops
import zipfile                 # Handle ZIP file extraction
import shutil                  # Copy files between folders
import xml.etree.ElementTree as ET  # XML utilities (not directly used but available if needed)

### Configuration

In [2]:
# Base URL for the Divvy S3 bucket listing (XML index of all available files)
BASE_URL = "https://divvy-tripdata.s3.amazonaws.com/?list-type=2"

# Root folder on local machine / Google Drive where all data will be stored
BASE_FOLDER = r"G:\My Drive\Divvy_TripData"

# Folder to store downloaded ZIP files
DOWNLOAD_FOLDER = os.path.join(
    BASE_FOLDER,
    "downloads"
)

# Folder to store extracted contents from the ZIP files
EXTRACT_FOLDER = os.path.join(
    BASE_FOLDER,
    "extracted"
)

# Folder to consolidate all CSV files from all months
CSV_FOLDER = os.path.join(
    BASE_FOLDER,
    "csv_master"
)

# Starting year and month for data to download
# Only files from this year/month onward will be considered
START_YEAR = 2024
START_MONTH = 1

### Create directories

In [3]:
# Ensure that all required folders exist before downloading or extracting
# If a folder already exists, os.makedirs with exist_ok=True will not raise an error
for folder in [
    DOWNLOAD_FOLDER,
    EXTRACT_FOLDER,
    CSV_FOLDER
]:
    os.makedirs(folder, exist_ok=True)

### Get available files from the website

In [4]:
# Connect to the Divvy S3 bucket and retrieve the XML listing of all objects
print("Connecting to Divvy S3 bucket...")

# Request the XML index of available files
response = requests.get(BASE_URL)

# Parse the XML response to extract file names (keys)
soup = BeautifulSoup(response.text, "xml")

# Build a list of all file keys (e.g., '202401-divvy-tripdata.zip', etc.)
files = [
    key.text
    for key in soup.find_all("Key")
]

# Display how many ZIP files were found in the bucket
print(f"Found {len(files)} ZIP files")

Connecting to Divvy S3 bucket...
Found 94 ZIP files


### Create date range for downloading files

In [5]:
# Get the current year and month to avoid requesting future data
current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

# List to hold the subset of files that fall within the desired date range
target_files = []

# Loop through each file key and select only those between START_YEAR/MONTH and current date
for file in files:
    try:
        # Extract year and month from the file name prefix (assumes 'YYYYMM...' format)
        date_part = file[:6]
        year = int(date_part[:4])
        month = int(date_part[4:6])

        # Include files from the start date forward
        if (
            (year > START_YEAR)
            or
            (year == START_YEAR and month >= START_MONTH)
        ):
            # Exclude files beyond the current year/month (future data)
            if (
                year < current_year
                or
                (year == current_year and month <= current_month)
            ):
                target_files.append(file)

    except:
        # If the file name does not match the expected date pattern, ignore it
        pass

# Sort the selected files to download them in chronological order
target_files.sort()

# Print how many files will actually be downloaded
print(
    f"Files to download: {len(target_files)}"
)

Files to download: 30


### Download ZIP files

In [6]:
# Iterate through filtered file list and download each ZIP file if not already present
for filename in tqdm(
    target_files,
    desc="Downloading files"
):
    # Build the full download URL for the current file
    url = (
        "https://divvy-tripdata.s3.amazonaws.com/"
        + filename
    )

    # Local path where the ZIP file will be stored
    zip_path = os.path.join(
        DOWNLOAD_FOLDER,
        filename
    )

    # Skip downloading if the file already exists locally
    if os.path.exists(zip_path):
        # Continue to the next file without re-downloading
        continue

    # Stream the file from the server to avoid loading it fully into memory
    with requests.get(
        url,
        stream=True
    ) as r:

        # Raise an exception if the request was not successful
        r.raise_for_status()

        # Get the total size from the headers (if available)
        total_size = int(
            r.headers.get(
                "content-length",
                0
            )
        )

        # Write the response content to disk in chunks
        with open(
            zip_path,
            "wb"
        ) as f:

            for chunk in r.iter_content(
                chunk_size=1024*1024
            ):
                # Write each chunk as it arrives
                f.write(chunk)

### Extract ZIP files

In [7]:
# Extract each downloaded ZIP file into its own subfolder
print("Extracting files...")

for filename in tqdm(
    target_files,
    desc="Extracting"
):

    # Full path to the downloaded ZIP file
    zip_path = os.path.join(
        DOWNLOAD_FOLDER,
        filename
    )

    # Folder where the contents of this ZIP will be extracted
    extract_path = os.path.join(
        EXTRACT_FOLDER,
        filename.replace(".zip","")
    )

    # Skip extraction if this ZIP has already been extracted
    if os.path.exists(extract_path):

        continue

    # Open the ZIP file and extract all contents into the target folder
    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            extract_path
        )

Extracting files...


Extracting: 100%|██████████| 30/30 [00:00<00:00, 272.82it/s]


### Move CSV files into a single directory

In [8]:
# Walk through the extracted folders and copy all CSV files into a single master folder
print("Collecting CSV files...")

for root, dirs, files in os.walk(EXTRACT_FOLDER):

    for file in files:

        # Skip macOS metadata files
        if file.startswith("._"):
            continue

        # Only copy real CSV files
        if file.endswith(".csv"):

            source = os.path.join(root, file)
            destination = os.path.join(CSV_FOLDER, file)

            shutil.copy2(source, destination)

print("\nCOMPLETE!")
print(f"CSV files stored in: {CSV_FOLDER}")


COMPLETE!
CSV files stored in: G:\My Drive\Divvy_TripData\csv_master


# Walk through the extracted folders and copy all CSV files into a single master folder
print("Collecting CSV files...")

for root, dirs, files in os.walk(
    EXTRACT_FOLDER
):

    for file in files:
        # Only process files with .csv extension
        if file.endswith(".csv"):
            # Full path to the source CSV
            source = os.path.join(
                root,
                file
            )

            # Destination path in the consolidated CSV folder
            destination = os.path.join(
                CSV_FOLDER,
                file
            )

            # Copy the CSV file while preserving metadata (timestamps, etc.)
            shutil.copy2(
                source,
                destination
            )

# Final status messages for the user
print("\nCOMPLETE!")
print(
    f"CSV files stored in: {CSV_FOLDER}"
)